In [25]:
import asyncio

import inspect

import math

import os

import re

import sys

import warnings

from pathlib import Path

from statistics import mean



LOCAL_PYDEPS = Path.cwd() / ".pydeps"

if LOCAL_PYDEPS.exists():

    sys.path.insert(0, str(LOCAL_PYDEPS))



import pandas as pd

import nltk

from dotenv import load_dotenv

try:

    import ragas.metrics.collections as ragas_metric_collections

except ImportError:

    ragas_metric_collections = None

import ragas.metrics as ragas_metrics_legacy

from langchain_community.retrievers import BM25Retriever

from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter



try:

    from langchain_core.prompts import ChatPromptTemplate

    from langchain_groq import ChatGroq

except ImportError:

    ChatPromptTemplate = None

    ChatGroq = None



try:

    from ragas.dataset_schema import SingleTurnSample

except ImportError:

    from ragas import SingleTurnSample



from nltk.tokenize import sent_tokenize, word_tokenize



load_dotenv()



for package in ("punkt", "punkt_tab"):

    try:

        nltk.download(package, quiet=True)

    except Exception:

        pass



BM25_K = 4

CHUNK_SIZE = 260

CHUNK_OVERLAP = 40

GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.1-8b-instant").strip() or "llama-3.1-8b-instant"



ANSWER_ENGINE = "extractive-fallback"

ANSWER_PROMPT = None

llm = None

groq_api_key = os.getenv("GROQ_API_KEY", "").strip()

if groq_api_key and ChatGroq is not None and ChatPromptTemplate is not None:

    try:

        llm = ChatGroq(api_key=groq_api_key, model=GROQ_MODEL, temperature=0)

        ANSWER_ENGINE = f"groq:{GROQ_MODEL}"

        ANSWER_PROMPT = ChatPromptTemplate.from_messages([

            (

                "system",

                "Answer only from the retrieved context. If the context is insufficient, say you do not know.",

            ),

            (

                "human",

                "Question:\n{question}\n\nRetrieved context:\n{context}\n\nGive a concise grounded answer.",

            ),

        ])

    except Exception:

        llm = None

        ANSWER_ENGINE = "extractive-fallback"

        ANSWER_PROMPT = None



def resolve_metric(metric_names: list[str]):

    metric_modules = []

    if ragas_metric_collections is not None:

        metric_modules.append(ragas_metric_collections)

    metric_modules.append(ragas_metrics_legacy)



    for metric_name in metric_names:

        for metric_module in metric_modules:

            with warnings.catch_warnings():

                warnings.simplefilter("ignore", DeprecationWarning)

                metric_cls = getattr(metric_module, metric_name, None)

            if metric_cls is not None:

                return metric_name, metric_cls()

    return None, None





def build_metric_kwargs(metric_callable, sample_payload: dict):

    signature = inspect.signature(metric_callable)

    accepts_var_kwargs = any(

        parameter.kind == inspect.Parameter.VAR_KEYWORD

        for parameter in signature.parameters.values()

    )

    if accepts_var_kwargs:

        return sample_payload

    return {

        name: value

        for name, value in sample_payload.items()

        if name in signature.parameters

    }





def score_ragas_metric(metric, sample: SingleTurnSample):

    if metric is None:

        return None

    if hasattr(metric, "single_turn_score"):

        return metric.single_turn_score(sample)

    if hasattr(metric, "single_turn_ascore"):

        return asyncio.run(metric.single_turn_ascore(sample))



    sample_payload = sample.model_dump(exclude_none=True) if hasattr(sample, "model_dump") else sample.dict(exclude_none=True)

    if hasattr(metric, "score"):

        metric_kwargs = build_metric_kwargs(metric.score, sample_payload)

        try:

            return metric.score(**metric_kwargs)

        except RuntimeError as exc:

            if "Use ascore() instead" not in str(exc) or not hasattr(metric, "ascore"):

                raise

    if hasattr(metric, "ascore"):

        metric_kwargs = build_metric_kwargs(metric.ascore, sample_payload)

        return asyncio.run(metric.ascore(**metric_kwargs))

    raise AttributeError(f"Unsupported Ragas metric API for {metric.__class__.__name__}")





def simplify_metric_value(value):

    if isinstance(value, dict):

        for preferred_key in ("rougeL", "rougeLsum", "f1", "score"):

            if preferred_key in value:

                return simplify_metric_value(value[preferred_key])

        return simplify_metric_value(next(iter(value.values())))

    if isinstance(value, (list, tuple)) and value:

        return simplify_metric_value(value[0])

    if value is None:

        return None

    try:

        return float(value)

    except (TypeError, ValueError):

        return value





GENERATION_METRIC_SPECS = {

    "exact_match": ["ExactMatch"],

    "bleu": ["BleuScore"],

    "rouge": ["RougeScore"],

}

RETRIEVAL_METRIC_SPECS = {

    "ragas_id_precision": ["IDBasedContextPrecision"],

    "ragas_id_recall": ["IDBasedContextRecall"],

}



generation_metrics = {}

for label, names in GENERATION_METRIC_SPECS.items():

    resolved_name, metric = resolve_metric(names)

    generation_metrics[label] = {"resolved_name": resolved_name, "metric": metric}



retrieval_metrics = {}

for label, names in RETRIEVAL_METRIC_SPECS.items():

    resolved_name, metric = resolve_metric(names)

    retrieval_metrics[label] = {"resolved_name": resolved_name, "metric": metric}



print("Retriever under evaluation:", "bm25_only")

print("Answer engine:", ANSWER_ENGINE)

print("Generation metrics:", {label: spec['resolved_name'] for label, spec in generation_metrics.items()})

print("Retrieval metrics:", {label: spec['resolved_name'] for label, spec in retrieval_metrics.items()})


Retriever under evaluation: bm25_only
Answer engine: groq:llama-3.3-70b-versatile
Generation metrics: {'exact_match': 'ExactMatch', 'bleu': 'BleuScore', 'rouge': 'RougeScore'}
Retrieval metrics: {'ragas_id_precision': 'IDBasedContextPrecision', 'ragas_id_recall': 'IDBasedContextRecall'}


In [33]:
RAW_SOURCES = [

    {

        "source": "invoice_aging_schema.txt",

        "topic": "exact_schema",

        "text": """

        The mart.finance.invoice_aging_daily table stores invoice aging snapshots.

        The overdue bucket for invoices more than ninety days late is AR_90_PLUS.

        The customer key is cust_id and the invoice key is invoice_id.

        Analysts filter AR_90_PLUS when they want the most severe unpaid balances.

        """

    },

    {

        "source": "collections_playbook.txt",

        "topic": "business_language",

        "text": """

        To identify chronic late payers, analysts look for repeated payment delays, old unpaid balances, and severe delinquency.

        Collections teams call chronic late payers the accounts that repeatedly miss due dates and keep balances open for long periods.

        Business users often ask how to identify chronic late payers when they do not know the warehouse field name AR_90_PLUS.

        """

    },

    {

        "source": "pipeline_runbook.txt",

        "topic": "pipeline_ops",

        "text": """

        The invoice aging mart is refreshed by job_7841 every morning at 05:00 UTC.

        Which job refreshes the invoice aging mart? The answer is job_7841.

        For chronic late payer reporting, inspect job_7841 first because it refreshes the invoice aging mart.

        If the mart is stale, operators first inspect job_7841 and then validate row counts in mart.finance.invoice_aging_daily.

        Exact identifiers such as job_7841 and mart.finance.invoice_aging_daily are important for keyword retrieval.

        """

    },

    {

        "source": "escalation_guidelines.txt",

        "topic": "collections_actions",

        "text": """

        Accounts should be escalated when invoices remain unpaid, promises to pay are broken, and overdue balances keep rolling forward month after month.

        Severe delinquency usually overlaps with the oldest aging bucket and repeated late payment behavior.

        """

    },

    {

        "source": "refund_policy.txt",

        "topic": "noise",

        "text": """

        Store credits can be requested within thirty days of purchase if an amount is incorrect.

        Refund operations belong to post-purchase support and are unrelated to receivables monitoring or collector workflows.

        """

    },

    {

        "source": "shipping_updates.txt",

        "topic": "noise",

        "text": """

        Shipping alerts are generated from logistics systems and do not depend on finance operations pipelines.

        Delivery exceptions and route changes are unrelated to receivables buckets or collector workflows.

        """

    },

]



RAW_EVAL_SET = [

    {

        "id": "q_refresh_job",

        "query": "Which job refreshes the invoice aging mart?",

        "gold_sources": ["pipeline_runbook.txt"],

        "expected_top_source": "pipeline_runbook.txt",

        "expected_source_order": ["pipeline_runbook.txt"],

        "required_answer_facts": ["job_7841"],

        "forbidden_answer_facts": ["refund", "shipping"],

        "reference_answer": "The invoice aging mart is refreshed by job_7841.",

    },

    {

        "id": "q_chronic_late_payers",

        "query": "How can analysts identify chronic late payers?",

        "gold_sources": ["collections_playbook.txt", "invoice_aging_schema.txt", "escalation_guidelines.txt"],

        "expected_top_source": "collections_playbook.txt",

        "expected_source_order": ["collections_playbook.txt", "invoice_aging_schema.txt", "escalation_guidelines.txt"],

        "required_answer_facts": ["repeated payment delays", "old unpaid balances", "severe delinquency", "AR_90_PLUS"],

        "forbidden_answer_facts": ["refund", "shipping"],

        "reference_answer": "Analysts identify chronic late payers by looking for repeated payment delays, old unpaid balances, severe delinquency, and the AR_90_PLUS overdue bucket.",

    },

    {

        "id": "q_overdue_field",

        "query": "Which field represents invoices more than 90 days overdue?",

        "gold_sources": ["invoice_aging_schema.txt"],

        "expected_top_source": "invoice_aging_schema.txt",

        "expected_source_order": ["invoice_aging_schema.txt"],

        "required_answer_facts": ["AR_90_PLUS"],

        "forbidden_answer_facts": ["refund", "shipping"],

        "reference_answer": "The field or bucket is AR_90_PLUS.",

    },

    {

        "id": "q_job_to_inspect",

        "query": "For chronic late payer reporting, which job should I inspect first?",

        "gold_sources": ["pipeline_runbook.txt", "collections_playbook.txt"],

        "expected_top_source": "pipeline_runbook.txt",

        "expected_source_order": ["pipeline_runbook.txt", "collections_playbook.txt"],

        "required_answer_facts": ["job_7841", "invoice aging mart", "severe delinquency"],

        "forbidden_answer_facts": ["refund", "shipping"],

        "reference_answer": "For chronic late payer reporting, inspect job_7841 first because it refreshes the invoice aging mart used to find severe delinquency.",

    },

]



splitter = RecursiveCharacterTextSplitter(

    chunk_size=CHUNK_SIZE,

    chunk_overlap=CHUNK_OVERLAP,

    separators=["\n\n", "\n", ". ", " "],

)



lc_documents = []

for item in RAW_SOURCES:

    for chunk_index, chunk_text in enumerate(splitter.split_text(item["text"].strip())):

        lc_documents.append(

            Document(

                page_content=chunk_text.strip(),

                metadata={

                    "chunk_id": f"{item['source']}::chunk_{chunk_index}",

                    "source": item["source"],

                    "topic": item["topic"],

                },

            )

        )



bm25_retriever = BM25Retriever.from_documents(lc_documents)

bm25_retriever.k = BM25_K

chunk_lookup = {doc.metadata["chunk_id"]: doc for doc in lc_documents}

source_to_chunk_ids = {}

for doc in lc_documents:

    source_to_chunk_ids.setdefault(doc.metadata["source"], []).append(doc.metadata["chunk_id"])



EVAL_SET = []

for item in RAW_EVAL_SET:

    gold_chunk_ids = [

        chunk_id

        for source in item["gold_sources"]

        for chunk_id in source_to_chunk_ids[source]

    ]

    EVAL_SET.append({

        **item,

        "gold_chunk_ids": gold_chunk_ids,

        "expected_top_chunk_id": source_to_chunk_ids[item["expected_top_source"]][0],

    })



preview = pd.DataFrame([

    {

        "chunk_id": doc.metadata["chunk_id"],

        "source": doc.metadata["source"],

        "topic": doc.metadata["topic"],

        "text": doc.page_content,

    }

    for doc in lc_documents

])



gt_preview = pd.DataFrame([

    {

        "id": item["id"],

        "query": item["query"],

        "gold_sources": " | ".join(item["gold_sources"]),

        "expected_top_source": item["expected_top_source"],

        "required_answer_facts": " | ".join(item["required_answer_facts"]),

        "forbidden_answer_facts": " | ".join(item["forbidden_answer_facts"]),

    }

    for item in EVAL_SET

])



print(f"Built {len(lc_documents)} chunks from {len(RAW_SOURCES)} synthetic source documents.")

print("Retriever under evaluation: bm25_only")

print("Ground-truth spec:")

display(gt_preview)

print("Chunk preview:")

display(preview)


Built 10 chunks from 6 synthetic source documents.
Retriever under evaluation: bm25_only
Ground-truth spec:


,id,query,gold_sources,expected_top_source,required_answer_facts,forbidden_answer_facts
0,q_refresh_job,Which job refreshes the invoice aging mart?,pipeline_runbook.txt,pipeline_runbook.txt,job_7841,refund | shipping
1,q_chronic_late_payers,How can analysts identify chronic late payers?,collections_playbook.txt | invoice_aging_schema.txt | escalation_guidelines.txt,collections_playbook.txt,repeated payment delays | old unpaid balances | severe delinquency | AR_90_PLUS,refund | shipping
2,q_overdue_field,Which field represents invoices more than 90 days overdue?,invoice_aging_schema.txt,invoice_aging_schema.txt,AR_90_PLUS,refund | shipping
3,q_job_to_inspect,"For chronic late payer reporting, which job should I inspect first?",pipeline_runbook.txt | collections_playbook.txt,pipeline_runbook.txt,job_7841 | invoice aging mart | severe delinquency,refund | shipping


Chunk preview:


,chunk_id,source,topic,text
0,invoice_aging_schema.txt::chunk_0,invoice_aging_schema.txt,exact_schema,The mart.finance.invoice_aging_daily table stores invoice aging snapshots.\n\n The overdue bucket for invoices more than ninety days late is AR_90_PL...
1,invoice_aging_schema.txt::chunk_1,invoice_aging_schema.txt,exact_schema,Analysts filter AR_90_PLUS when they want the most severe unpaid balances.
2,collections_playbook.txt::chunk_0,collections_playbook.txt,business_language,"To identify chronic late payers, analysts look for repeated payment delays, old unpaid balances, and severe delinquency.\n\n Collections teams call c..."
3,collections_playbook.txt::chunk_1,collections_playbook.txt,business_language,Business users often ask how to identify chronic late payers when they do not know the warehouse field name AR_90_PLUS.
4,pipeline_runbook.txt::chunk_0,pipeline_runbook.txt,pipeline_ops,The invoice aging mart is refreshed by job_7841 every morning at 05:00 UTC.\n\n Which job refreshes the invoice aging mart? The answer is job_7841.
5,pipeline_runbook.txt::chunk_1,pipeline_runbook.txt,pipeline_ops,"For chronic late payer reporting, inspect job_7841 first because it refreshes the invoice aging mart.\n\n If the mart is stale, operators first inspe..."
6,pipeline_runbook.txt::chunk_2,pipeline_runbook.txt,pipeline_ops,Exact identifiers such as job_7841 and mart.finance.invoice_aging_daily are important for keyword retrieval.
7,escalation_guidelines.txt::chunk_0,escalation_guidelines.txt,collections_actions,"Accounts should be escalated when invoices remain unpaid, promises to pay are broken, and overdue balances keep rolling forward month after month.\n\n ..."
8,refund_policy.txt::chunk_0,refund_policy.txt,noise,Store credits can be requested within thirty days of purchase if an amount is incorrect.\n\n Refund operations belong to post-purchase support and ar...
9,shipping_updates.txt::chunk_0,shipping_updates.txt,noise,Shipping alerts are generated from logistics systems and do not depend on finance operations pipelines.\n\n Delivery exceptions and route changes are...


In [34]:
if "bm25_retriever" not in globals() or "EVAL_SET" not in globals():

    raise RuntimeError("Run Cell 2 first.")



EVALUATED_PIPELINE = "bm25_only"

RETRIEVAL_K = BM25_K



def normalize_tokens(text: str) -> list[str]:

    return [

        token

        for token in word_tokenize(text.lower())

        if re.search(r"[a-z0-9_]", token)

    ]





def normalize_text(text: str) -> str:

    return " ".join(normalize_tokens(text))





def retrieve_context(query: str):

    return bm25_retriever.invoke(query)





def ranked_sources_text(docs: list[Document]) -> str:

    return " | ".join(

        f"{rank}. {doc.metadata['source']}::{doc.metadata['chunk_id'].split('::')[-1]}"

        for rank, doc in enumerate(docs, start=1)

    )





def chunk_precision_at_k(retrieved_ids, gold_ids, k: int = RETRIEVAL_K) -> float:

    top_ids = retrieved_ids[:k]

    if not top_ids:

        return 0.0

    gold_set = set(gold_ids)

    hits = sum(1 for chunk_id in top_ids if chunk_id in gold_set)

    return hits / len(top_ids)





def chunk_recall_at_k(retrieved_ids, gold_ids, k: int = RETRIEVAL_K) -> float:

    gold_set = set(gold_ids)

    if not gold_set:

        return 0.0

    return len(set(retrieved_ids[:k]) & gold_set) / len(gold_set)





def source_precision_at_k(retrieved_docs, gold_sources, k: int = RETRIEVAL_K) -> float:

    top_sources = [doc.metadata["source"] for doc in retrieved_docs[:k]]

    if not top_sources:

        return 0.0

    gold_set = set(gold_sources)

    hits = sum(1 for source in top_sources if source in gold_set)

    return hits / len(top_sources)





def source_recall_at_k(retrieved_docs, gold_sources, k: int = RETRIEVAL_K) -> float:

    gold_set = set(gold_sources)

    if not gold_set:

        return 0.0

    fetched_sources = {doc.metadata["source"] for doc in retrieved_docs[:k]}

    return len(fetched_sources & gold_set) / len(gold_set)





def all_gold_sources_fetched(retrieved_docs, gold_sources, k: int = RETRIEVAL_K) -> float:

    fetched_sources = {doc.metadata["source"] for doc in retrieved_docs[:k]}

    return float(set(gold_sources).issubset(fetched_sources))





def expected_top_source_hit(retrieved_docs, expected_top_source: str) -> float:

    return float(bool(retrieved_docs) and retrieved_docs[0].metadata["source"] == expected_top_source)





def first_relevant_rank(retrieved_ids, gold_ids) -> int:

    gold_set = set(gold_ids)

    for rank, chunk_id in enumerate(retrieved_ids, start=1):

        if chunk_id in gold_set:

            return rank

    return len(retrieved_ids) + 1





def reciprocal_rank(retrieved_ids, gold_ids) -> float:

    gold_set = set(gold_ids)

    for rank, chunk_id in enumerate(retrieved_ids, start=1):

        if chunk_id in gold_set:

            return 1.0 / rank

    return 0.0





def ndcg_at_k(retrieved_ids, gold_ids, k: int = RETRIEVAL_K) -> float:

    gold_set = set(gold_ids)

    dcg = 0.0

    for rank, chunk_id in enumerate(retrieved_ids[:k], start=1):

        if chunk_id in gold_set:

            dcg += 1.0 / math.log2(rank + 1)

    ideal_hits = min(len(gold_set), k)

    ideal_dcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))

    return dcg / ideal_dcg if ideal_dcg else 0.0





def lexical_overlap(query: str, doc: Document) -> float:

    query_tokens = set(normalize_tokens(query))

    doc_tokens = set(normalize_tokens(doc.page_content))

    if not query_tokens:

        return 0.0

    return len(query_tokens & doc_tokens) / len(query_tokens)





def ragas_retrieval_scores(retrieved_ids, gold_ids):

    sample = SingleTurnSample(

        user_input="retrieval-eval",

        retrieved_context_ids=retrieved_ids,

        reference_context_ids=gold_ids,

    )

    scores = {}

    for label, spec in retrieval_metrics.items():

        value = score_ragas_metric(spec["metric"], sample)

        scores[label] = simplify_metric_value(value)

    return scores





retrieval_runs = {}

retrieval_rows = []

retrieval_trace_rows = []

for example in EVAL_SET:

    retrieved_docs = retrieve_context(example["query"])

    retrieved_ids = [doc.metadata["chunk_id"] for doc in retrieved_docs]

    retrieval_runs[example["id"]] = {

        "docs": retrieved_docs,

        "ids": retrieved_ids,

    }

    ragas_scores = ragas_retrieval_scores(retrieved_ids, example["gold_chunk_ids"])



    retrieval_rows.append({

        "id": example["id"],

        "query": example["query"],

        "fetch_source_precision_at_k": source_precision_at_k(retrieved_docs, example["gold_sources"], k=RETRIEVAL_K),

        "fetch_source_recall_at_k": source_recall_at_k(retrieved_docs, example["gold_sources"], k=RETRIEVAL_K),

        "fetch_chunk_precision_at_k": chunk_precision_at_k(retrieved_ids, example["gold_chunk_ids"], k=RETRIEVAL_K),

        "fetch_chunk_recall_at_k": chunk_recall_at_k(retrieved_ids, example["gold_chunk_ids"], k=RETRIEVAL_K),

        "all_gold_sources_fetched": all_gold_sources_fetched(retrieved_docs, example["gold_sources"], k=RETRIEVAL_K),

        "top_1_source_correct": expected_top_source_hit(retrieved_docs, example["expected_top_source"]),

        "first_relevant_rank": first_relevant_rank(retrieved_ids, example["gold_chunk_ids"]),

        "mrr": reciprocal_rank(retrieved_ids, example["gold_chunk_ids"]),

        "ndcg_at_k": ndcg_at_k(retrieved_ids, example["gold_chunk_ids"], k=RETRIEVAL_K),

        "avg_query_token_overlap": mean([lexical_overlap(example["query"], doc) for doc in retrieved_docs]) if retrieved_docs else 0.0,

        **ragas_scores,

    })



    retrieval_trace_rows.append({

        "id": example["id"],

        "query": example["query"],

        "gold_sources": " | ".join(example["gold_sources"]),

        "expected_top_source": example["expected_top_source"],

        "top_1_source": retrieved_docs[0].metadata["source"] if retrieved_docs else "",

        "ranked_sources": ranked_sources_text(retrieved_docs),

    })



retrieval_df = pd.DataFrame(retrieval_rows)

retrieval_trace_df = pd.DataFrame(retrieval_trace_rows)



manual_fetch_summary = retrieval_df[[

    "fetch_source_precision_at_k",

    "fetch_source_recall_at_k",

    "fetch_chunk_precision_at_k",

    "fetch_chunk_recall_at_k",

    "all_gold_sources_fetched",

]].mean(numeric_only=True).to_frame().T

manual_fetch_summary.insert(0, "pipeline", EVALUATED_PIPELINE)

manual_fetch_summary.insert(1, "queries", len(EVAL_SET))

manual_fetch_summary.insert(2, "k", RETRIEVAL_K)



manual_rank_summary = retrieval_df[[

    "top_1_source_correct",

    "first_relevant_rank",

    "mrr",

    "ndcg_at_k",

    "avg_query_token_overlap",

]].mean(numeric_only=True).to_frame().T

manual_rank_summary.insert(0, "pipeline", EVALUATED_PIPELINE)

manual_rank_summary.insert(1, "queries", len(EVAL_SET))

manual_rank_summary.insert(2, "k", RETRIEVAL_K)



ragas_retrieval_summary = retrieval_df[[

    "ragas_id_precision",

    "ragas_id_recall",

]].mean(numeric_only=True).to_frame().T

ragas_retrieval_summary.insert(0, "pipeline", EVALUATED_PIPELINE)

ragas_retrieval_summary.insert(1, "queries", len(EVAL_SET))

ragas_retrieval_summary.insert(2, "k", RETRIEVAL_K)



pd.set_option("display.max_colwidth", 160)

print(f"Pipeline under evaluation: {EVALUATED_PIPELINE}")

print("Fetch-level manual summary:")

display(manual_fetch_summary.round(3))

print("Rank-level manual summary:")

display(manual_rank_summary.round(3))

print("Retrieval Ragas summary:")

display(ragas_retrieval_summary.round(3))

print("Per-query retrieval evaluation:")

display(retrieval_df.round(3))

print("Retrieval ranking trace:")

display(retrieval_trace_df)


Pipeline under evaluation: bm25_only
Fetch-level manual summary:


,pipeline,queries,k,fetch_source_precision_at_k,fetch_source_recall_at_k,fetch_chunk_precision_at_k,fetch_chunk_recall_at_k,all_gold_sources_fetched
0,bm25_only,4,4,0.5,0.833,0.5,0.542,0.75


Rank-level manual summary:


,pipeline,queries,k,top_1_source_correct,first_relevant_rank,mrr,ndcg_at_k,avg_query_token_overlap
0,bm25_only,4,4,1.0,1.0,1.0,0.705,0.403


Retrieval Ragas summary:


,pipeline,queries,k,ragas_id_precision,ragas_id_recall
0,bm25_only,4,4,0.5,0.542


Per-query retrieval evaluation:


,id,query,fetch_source_precision_at_k,fetch_source_recall_at_k,fetch_chunk_precision_at_k,fetch_chunk_recall_at_k,all_gold_sources_fetched,top_1_source_correct,first_relevant_rank,mrr,ndcg_at_k,avg_query_token_overlap,ragas_id_precision,ragas_id_recall
0,q_refresh_job,Which job refreshes the invoice aging mart?,0.50,1.000,0.50,0.667,1.0,1.0,1,1.0,0.765,0.607,0.50,0.667
1,q_chronic_late_payers,How can analysts identify chronic late payers?,0.50,0.333,0.50,0.400,0.0,1.0,1,1.0,0.637,0.464,0.50,0.400
2,q_overdue_field,Which field represents invoices more than 90 days overdue?,0.25,1.000,0.25,0.500,1.0,1.0,1,1.0,0.613,0.222,0.25,0.500
3,q_job_to_inspect,"For chronic late payer reporting, which job should I inspect first?",0.75,1.000,0.75,0.600,1.0,1.0,1,1.0,0.805,0.318,0.75,0.600


Retrieval ranking trace:


,id,query,gold_sources,expected_top_source,top_1_source,ranked_sources
0,q_refresh_job,Which job refreshes the invoice aging mart?,pipeline_runbook.txt,pipeline_runbook.txt,pipeline_runbook.txt,1. pipeline_runbook.txt::chunk_0 | 2. pipeline_runbook.txt::chunk_1 | 3. invoice_aging_schema.txt::chunk_0 | 4. escalation_guidelines.txt::chunk_0
1,q_chronic_late_payers,How can analysts identify chronic late payers?,collections_playbook.txt | invoice_aging_schema.txt | escalation_guidelines.txt,collections_playbook.txt,collections_playbook.txt,1. collections_playbook.txt::chunk_0 | 2. collections_playbook.txt::chunk_1 | 3. refund_policy.txt::chunk_0 | 4. pipeline_runbook.txt::chunk_1
2,q_overdue_field,Which field represents invoices more than 90 days overdue?,invoice_aging_schema.txt,invoice_aging_schema.txt,invoice_aging_schema.txt,1. invoice_aging_schema.txt::chunk_0 | 2. collections_playbook.txt::chunk_1 | 3. pipeline_runbook.txt::chunk_0 | 4. refund_policy.txt::chunk_0
3,q_job_to_inspect,"For chronic late payer reporting, which job should I inspect first?",pipeline_runbook.txt | collections_playbook.txt,pipeline_runbook.txt,pipeline_runbook.txt,1. pipeline_runbook.txt::chunk_1 | 2. pipeline_runbook.txt::chunk_0 | 3. escalation_guidelines.txt::chunk_0 | 4. collections_playbook.txt::chunk_0


In [39]:
if "retrieval_runs" not in globals() or "EVALUATED_PIPELINE" not in globals():

    raise RuntimeError("Run Cell 3 first.")



import json

from pydantic import BaseModel, Field



def build_context(docs) -> str:

    return "\n\n".join(

        f"Source: {doc.metadata['source']}\n{doc.page_content}"

        for doc in docs

    )





def phrase_present(text: str, phrase: str) -> bool:

    return normalize_text(phrase) in normalize_text(text)





def token_overlap_ratio(source_text: str, reference_text: str) -> float:

    source_tokens = set(normalize_tokens(source_text))

    reference_tokens = set(normalize_tokens(reference_text))

    if not source_tokens:

        return 0.0

    return len(source_tokens & reference_tokens) / len(source_tokens)





def answer_with_extractive_rag(query: str, docs, max_sentences: int = 2) -> str:

    query_tokens = set(normalize_tokens(query))

    scored_sentences = []

    for doc in docs:

        for sentence in sent_tokenize(doc.page_content):

            sentence_tokens = set(normalize_tokens(sentence))

            lexical_score = len(query_tokens & sentence_tokens)

            exact_bonus = sum(1 for token in query_tokens if token in sentence.lower() and ("_" in token or token.isdigit()))

            score = lexical_score + (2 * exact_bonus)

            if score > 0:

                scored_sentences.append((score, sentence.strip()))

    if not scored_sentences:

        return "I do not know from the retrieved context."

    ordered_sentences = []

    seen = set()

    for _, sentence in sorted(scored_sentences, key=lambda item: item[0], reverse=True):

        if sentence not in seen:

            seen.add(sentence)

            ordered_sentences.append(sentence)

        if len(ordered_sentences) == max_sentences:

            break

    return " ".join(ordered_sentences)





def answer_with_bm25_rag(query: str, docs):

    context = build_context(docs)

    if llm is not None and ANSWER_PROMPT is not None:

        try:

            response = (ANSWER_PROMPT | llm).invoke({

                "question": query,

                "context": context,

            })

            response_text = response.content if hasattr(response, "content") else str(response)

            return response_text.strip(), ANSWER_ENGINE

        except Exception:

            pass

    return answer_with_extractive_rag(query, docs), "extractive-fallback"





def manual_answer_scores(answer: str, docs, example: dict):

    context = build_context(docs)

    required_hits = [fact for fact in example["required_answer_facts"] if phrase_present(answer, fact)]

    missing_required_facts = [fact for fact in example["required_answer_facts"] if fact not in required_hits]

    forbidden_hits = [fact for fact in example["forbidden_answer_facts"] if phrase_present(answer, fact)]

    return {

        "manual_exact_match": float(normalize_text(answer) == normalize_text(example["reference_answer"])),

        "required_fact_recall": len(required_hits) / len(example["required_answer_facts"]) if example["required_answer_facts"] else 1.0,

        "all_required_facts_present": float(len(required_hits) == len(example["required_answer_facts"])),

        "forbidden_fact_hits": float(len(forbidden_hits)),

        "context_support_of_required_facts": sum(1 for fact in example["required_answer_facts"] if phrase_present(context, fact)) / len(example["required_answer_facts"] if example["required_answer_facts"] else [1]),

        "answer_context_overlap": token_overlap_ratio(answer, context),

        "missing_required_facts": missing_required_facts,

        "forbidden_fact_list": forbidden_hits,

    }





def ragas_generation_scores(answer: str, reference: str):

    sample = SingleTurnSample(

        user_input="generation-eval",

        response=answer,

        reference=reference,

    )

    scores = {}

    for label, spec in generation_metrics.items():

        value = score_ragas_metric(spec["metric"], sample)

        scores[f"ragas_{label}"] = simplify_metric_value(value)

    return scores





class JudgeResult(BaseModel):

    groundedness_score: float = Field(ge=1, le=5)

    reference_alignment_score: float = Field(ge=1, le=5)

    completeness_score: float = Field(ge=1, le=5)

    reason: str





JUDGE_PROMPT = None

JUDGE_STRUCTURED_LLM = None

if llm is not None and ChatPromptTemplate is not None:

    JUDGE_PROMPT = ChatPromptTemplate.from_messages([

        (

            "system",

            "You are a strict RAG evaluator. Score the candidate answer using only the retrieved context and the reference answer.",

        ),

        (

            "human",

            "Question:\n{question}\n\nRetrieved context:\n{context}\n\nReference answer:\n{reference_answer}\n\nRequired facts:\n{required_facts}\n\nForbidden facts:\n{forbidden_facts}\n\nCandidate answer:\n{candidate_answer}\n\nScore groundedness, reference alignment, and completeness from 1 to 5, then explain briefly.",

        ),

    ])

    try:

        JUDGE_STRUCTURED_LLM = llm.with_structured_output(JudgeResult)

    except Exception:

        JUDGE_STRUCTURED_LLM = None





def parse_judge_payload(raw_text: str) -> dict:

    match = re.search(r"\{.*\}", raw_text, flags=re.S)

    if match:

        try:

            return json.loads(match.group(0))

        except json.JSONDecodeError:

            pass



    extracted = {}

    patterns = {

        "groundedness_score": r"grounded(?:ness)?(?:_score)?\s*[:=-]\s*([1-5](?:\.\d+)?)",

        "reference_alignment_score": r"reference(?:\s+alignment)?(?:_score)?\s*[:=-]\s*([1-5](?:\.\d+)?)",

        "completeness_score": r"completeness(?:_score)?\s*[:=-]\s*([1-5](?:\.\d+)?)",

    }

    for key, pattern in patterns.items():

        matched = re.search(pattern, raw_text, flags=re.I)

        if matched:

            extracted[key] = matched.group(1)

    if extracted:

        extracted["reason"] = raw_text.strip()[:300]

    return extracted





def coerce_judge_score(value):

    try:

        numeric_value = float(value)

    except (TypeError, ValueError):

        return None

    return min(5.0, max(1.0, numeric_value))





def llm_judge_scores(query: str, docs, answer: str, example: dict):

    if llm is None or JUDGE_PROMPT is None:

        return {

            "llm_judge_groundedness": None,

            "llm_judge_reference_alignment": None,

            "llm_judge_completeness": None,

            "llm_judge_average": None,

            "llm_judge_reason": "judge unavailable",

        }



    judge_inputs = {

        "question": query,

        "context": build_context(docs),

        "reference_answer": example["reference_answer"],

        "required_facts": " | ".join(example["required_answer_facts"]),

        "forbidden_facts": " | ".join(example["forbidden_answer_facts"]),

        "candidate_answer": answer,

    }



    payload = {}

    raw_reason = "judge response unavailable"

    if JUDGE_STRUCTURED_LLM is not None:

        try:

            structured_result = (JUDGE_PROMPT | JUDGE_STRUCTURED_LLM).invoke(judge_inputs)

            payload = structured_result.model_dump() if hasattr(structured_result, "model_dump") else dict(structured_result)

        except Exception:

            payload = {}



    if not payload:

        try:

            response = (JUDGE_PROMPT | llm).invoke(judge_inputs)

            response_text = response.content if hasattr(response, "content") else str(response)

            payload = parse_judge_payload(response_text)

            raw_reason = response_text.strip()[:300]

        except Exception as exc:

            return {

                "llm_judge_groundedness": None,

                "llm_judge_reference_alignment": None,

                "llm_judge_completeness": None,

                "llm_judge_average": None,

                "llm_judge_reason": f"judge error: {exc}",

            }



    groundedness = coerce_judge_score(payload.get("groundedness_score"))

    reference_alignment = coerce_judge_score(payload.get("reference_alignment_score"))

    completeness = coerce_judge_score(payload.get("completeness_score"))

    numeric_scores = [score for score in [groundedness, reference_alignment, completeness] if score is not None]

    average_score = mean(numeric_scores) if numeric_scores else None

    return {

        "llm_judge_groundedness": groundedness,

        "llm_judge_reference_alignment": reference_alignment,

        "llm_judge_completeness": completeness,

        "llm_judge_average": average_score,

        "llm_judge_reason": payload.get("reason", raw_reason),

    }





generation_rows = []

generation_example_rows = []

for example in EVAL_SET:

    retrieved_docs = retrieval_runs[example["id"]]["docs"]

    answer, used_answer_engine = answer_with_bm25_rag(example["query"], retrieved_docs)

    manual_scores = manual_answer_scores(answer, retrieved_docs, example)

    ragas_scores = ragas_generation_scores(answer, example["reference_answer"])

    judge_scores = llm_judge_scores(example["query"], retrieved_docs, answer, example)



    generation_rows.append({

        "id": example["id"],

        "query": example["query"],

        "used_answer_engine": used_answer_engine,

        "manual_exact_match": manual_scores["manual_exact_match"],

        "required_fact_recall": manual_scores["required_fact_recall"],

        "all_required_facts_present": manual_scores["all_required_facts_present"],

        "forbidden_fact_hits": manual_scores["forbidden_fact_hits"],

        "context_support_of_required_facts": manual_scores["context_support_of_required_facts"],

        "answer_context_overlap": manual_scores["answer_context_overlap"],

        **ragas_scores,

        "llm_judge_groundedness": judge_scores["llm_judge_groundedness"],

        "llm_judge_reference_alignment": judge_scores["llm_judge_reference_alignment"],

        "llm_judge_completeness": judge_scores["llm_judge_completeness"],

        "llm_judge_average": judge_scores["llm_judge_average"],

    })



    generation_example_rows.append({

        "id": example["id"],

        "query": example["query"],

        "used_answer_engine": used_answer_engine,

        "retrieved_sources": " | ".join(doc.metadata["source"] for doc in retrieved_docs),

        "generated_answer": answer,

        "reference_answer": example["reference_answer"],

        "missing_required_facts": " | ".join(manual_scores["missing_required_facts"]) or "-",

        "forbidden_fact_list": " | ".join(manual_scores["forbidden_fact_list"]) or "-",

        "llm_judge_reason": judge_scores["llm_judge_reason"],

    })



generation_df = pd.DataFrame(generation_rows)

generation_examples_df = pd.DataFrame(generation_example_rows)



for judge_column in [

    "llm_judge_groundedness",

    "llm_judge_reference_alignment",

    "llm_judge_completeness",

    "llm_judge_average",

]:

    generation_df[judge_column] = pd.to_numeric(generation_df[judge_column], errors="coerce")



manual_answer_summary = generation_df[[

    "manual_exact_match",

    "required_fact_recall",

    "all_required_facts_present",

    "forbidden_fact_hits",

    "context_support_of_required_facts",

    "answer_context_overlap",

]].mean(numeric_only=True).to_frame().T

manual_answer_summary.insert(0, "pipeline", EVALUATED_PIPELINE)

manual_answer_summary.insert(1, "queries", len(EVAL_SET))



ragas_answer_summary = generation_df[[

    "ragas_exact_match",

    "ragas_bleu",

    "ragas_rouge",

]].mean(numeric_only=True).to_frame().T

ragas_answer_summary.insert(0, "pipeline", EVALUATED_PIPELINE)

ragas_answer_summary.insert(1, "queries", len(EVAL_SET))



llm_judge_summary = generation_df[[

    "llm_judge_groundedness",

    "llm_judge_reference_alignment",

    "llm_judge_completeness",

    "llm_judge_average",

]].mean(numeric_only=True).to_frame().T

llm_judge_summary.insert(0, "pipeline", EVALUATED_PIPELINE)

llm_judge_summary.insert(1, "queries", len(EVAL_SET))



print(f"Pipeline under evaluation: {EVALUATED_PIPELINE}")

print("Answer engines used:", sorted(generation_df["used_answer_engine"].unique()))

print("Answer-level manual summary:")

display(manual_answer_summary.round(3))

print("Answer-level Ragas summary:")

display(ragas_answer_summary.round(3))

print("Answer-level LLM judge summary:")

display(llm_judge_summary.round(3))

print("Per-query answer evaluation:")

display(generation_df.round(3))

print("Answer examples and GT gaps:")

display(generation_examples_df)


Pipeline under evaluation: bm25_only
Answer engines used: ['groq:llama-3.3-70b-versatile']
Answer-level manual summary:


,pipeline,queries,manual_exact_match,required_fact_recall,all_required_facts_present,forbidden_fact_hits,context_support_of_required_facts,answer_context_overlap
0,bm25_only,4,0.0,0.771,0.5,0.0,1.0,0.845


Answer-level Ragas summary:


,pipeline,queries,ragas_exact_match,ragas_bleu,ragas_rouge
0,bm25_only,4,0.0,0.36,0.592


Answer-level LLM judge summary:


,pipeline,queries,llm_judge_groundedness,llm_judge_reference_alignment,llm_judge_completeness,llm_judge_average
0,bm25_only,4,4.75,4.5,4.25,4.5


Per-query answer evaluation:


,id,query,used_answer_engine,manual_exact_match,required_fact_recall,all_required_facts_present,forbidden_fact_hits,context_support_of_required_facts,answer_context_overlap,ragas_exact_match,ragas_bleu,ragas_rouge,llm_judge_groundedness,llm_judge_reference_alignment,llm_judge_completeness,llm_judge_average
0,q_refresh_job,Which job refreshes the invoice aging mart?,groq:llama-3.3-70b-versatile,0.0,1.000,1.0,0.0,1.0,0.889,0.0,0.398,0.700,5.0,5.0,5.0,5.000
1,q_chronic_late_payers,How can analysts identify chronic late payers?,groq:llama-3.3-70b-versatile,0.0,0.750,0.0,0.0,1.0,0.793,0.0,0.441,0.607,5.0,4.0,4.0,4.333
2,q_overdue_field,Which field represents invoices more than 90 days overdue?,groq:llama-3.3-70b-versatile,0.0,1.000,1.0,0.0,1.0,0.800,0.0,0.293,0.500,5.0,5.0,5.0,5.000
3,q_job_to_inspect,"For chronic late payer reporting, which job should I inspect first?",groq:llama-3.3-70b-versatile,0.0,0.333,0.0,0.0,1.0,0.900,0.0,0.308,0.562,4.0,4.0,3.0,3.667


Answer examples and GT gaps:


,id,query,used_answer_engine,retrieved_sources,generated_answer,reference_answer,missing_required_facts,forbidden_fact_list,llm_judge_reason
0,q_refresh_job,Which job refreshes the invoice aging mart?,groq:llama-3.3-70b-versatile,pipeline_runbook.txt | pipeline_runbook.txt | invoice_aging_schema.txt | escalation_guidelines.txt,The job that refreshes the invoice aging mart is job_7841.,The invoice aging mart is refreshed by job_7841.,-,-,"The candidate answer directly matches the reference answer and is supported by the retrieved context, which explicitly states that job_7841 refreshes the in..."
1,q_chronic_late_payers,How can analysts identify chronic late payers?,groq:llama-3.3-70b-versatile,collections_playbook.txt | collections_playbook.txt | refund_policy.txt | pipeline_runbook.txt,"Analysts can identify chronic late payers by looking for repeated payment delays, old unpaid balances, and severe delinquency, which are characterized by re...","Analysts identify chronic late payers by looking for repeated payment delays, old unpaid balances, severe delinquency, and the AR_90_PLUS overdue bucket.",AR_90_PLUS,-,"The candidate answer is well-grounded in the context and accurately identifies key characteristics of chronic late payers, but misses the specific mention o..."
2,q_overdue_field,Which field represents invoices more than 90 days overdue?,groq:llama-3.3-70b-versatile,invoice_aging_schema.txt | collections_playbook.txt | pipeline_runbook.txt | refund_policy.txt,The field AR_90_PLUS represents invoices more than 90 days overdue.,The field or bucket is AR_90_PLUS.,-,-,"The candidate answer directly matches the required fact AR_90_PLUS and aligns with the reference answer, providing a complete and accurate response."
3,q_job_to_inspect,"For chronic late payer reporting, which job should I inspect first?",groq:llama-3.3-70b-versatile,pipeline_runbook.txt | pipeline_runbook.txt | escalation_guidelines.txt | collections_playbook.txt,"For chronic late payer reporting, you should inspect job_7841 first.","For chronic late payer reporting, inspect job_7841 first because it refreshes the invoice aging mart used to find severe delinquency.",invoice aging mart | severe delinquency,-,"The candidate answer is partially complete and grounded in the context, but lacks the specific reason for inspecting job_7841, which is that it refreshes th..."


In [40]:
if "manual_fetch_summary" not in globals() or "manual_answer_summary" not in globals():

    raise RuntimeError("Run Cells 3 and 4 first.")



fetch_risk_row = retrieval_df.sort_values(

    ["fetch_source_recall_at_k", "fetch_chunk_recall_at_k", "all_gold_sources_fetched"],

    ascending=[True, True, True],

).iloc[0]



rank_risk_row = retrieval_df.sort_values(

    ["top_1_source_correct", "mrr", "ndcg_at_k"],

    ascending=[True, True, True],

).iloc[0]



answer_risk_row = generation_df.sort_values(

    ["required_fact_recall", "ragas_rouge", "llm_judge_average"],

    ascending=[True, True, True],

).iloc[0]



level_metric_rows = []

for level_name, summary_df, drop_columns in [

    ("fetch_manual", manual_fetch_summary, ["pipeline", "queries", "k"]),

    ("rank_manual", manual_rank_summary, ["pipeline", "queries", "k"]),

    ("retrieval_ragas", ragas_retrieval_summary, ["pipeline", "queries", "k"]),

    ("answer_manual", manual_answer_summary, ["pipeline", "queries"]),

    ("answer_ragas", ragas_answer_summary, ["pipeline", "queries"]),

    ("answer_llm_judge", llm_judge_summary, ["pipeline", "queries"]),

]:

    metric_values = summary_df.iloc[0].drop(drop_columns).to_dict()

    for metric_name, metric_value in metric_values.items():

        level_metric_rows.append({

            "level": level_name,

            "metric": metric_name,

            "value": metric_value,

        })



level_summary = pd.DataFrame(level_metric_rows)



overall_evaluation = pd.DataFrame([

    {

        "pipeline": EVALUATED_PIPELINE,

        "retriever": "BM25Retriever",

        "answer_engine": " | ".join(sorted(generation_df["used_answer_engine"].unique())),

        "fetch_source_recall_at_k": manual_fetch_summary.iloc[0]["fetch_source_recall_at_k"],

        "fetch_chunk_recall_at_k": manual_fetch_summary.iloc[0]["fetch_chunk_recall_at_k"],

        "top_1_source_correct": manual_rank_summary.iloc[0]["top_1_source_correct"],

        "mrr": manual_rank_summary.iloc[0]["mrr"],

        "ndcg_at_k": manual_rank_summary.iloc[0]["ndcg_at_k"],

        "ragas_id_precision": ragas_retrieval_summary.iloc[0]["ragas_id_precision"],

        "ragas_id_recall": ragas_retrieval_summary.iloc[0]["ragas_id_recall"],

        "required_fact_recall": manual_answer_summary.iloc[0]["required_fact_recall"],

        "forbidden_fact_hits": manual_answer_summary.iloc[0]["forbidden_fact_hits"],

        "answer_context_overlap": manual_answer_summary.iloc[0]["answer_context_overlap"],

        "ragas_exact_match": ragas_answer_summary.iloc[0]["ragas_exact_match"],

        "ragas_bleu": ragas_answer_summary.iloc[0]["ragas_bleu"],

        "ragas_rouge": ragas_answer_summary.iloc[0]["ragas_rouge"],

        "llm_judge_groundedness": llm_judge_summary.iloc[0]["llm_judge_groundedness"],

        "llm_judge_reference_alignment": llm_judge_summary.iloc[0]["llm_judge_reference_alignment"],

        "llm_judge_completeness": llm_judge_summary.iloc[0]["llm_judge_completeness"],

        "llm_judge_average": llm_judge_summary.iloc[0]["llm_judge_average"],

    }

])



print("Level-by-level evaluation summary:")

display(level_summary.round(3))

print("Complete evaluation snapshot:")

display(overall_evaluation.round(3))

print(f"Weakest fetch query: {fetch_risk_row['query']}")

print(f"Weakest rank query: {rank_risk_row['query']}")

print(f"Weakest answer query: {answer_risk_row['query']}")

print("This notebook now evaluates a BM25-only RAG pipeline with explicit ground truth at fetch, rank, and answer level using manual, Ragas, and LLM-judge metrics.")


Level-by-level evaluation summary:


,level,metric,value
0,fetch_manual,fetch_source_precision_at_k,0.500
1,fetch_manual,fetch_source_recall_at_k,0.833
2,fetch_manual,fetch_chunk_precision_at_k,0.500
3,fetch_manual,fetch_chunk_recall_at_k,0.542
4,fetch_manual,all_gold_sources_fetched,0.750
5,rank_manual,top_1_source_correct,1.000
6,rank_manual,first_relevant_rank,1.000
7,rank_manual,mrr,1.000
8,rank_manual,ndcg_at_k,0.705
9,rank_manual,avg_query_token_overlap,0.403


Complete evaluation snapshot:


,pipeline,retriever,answer_engine,fetch_source_recall_at_k,fetch_chunk_recall_at_k,top_1_source_correct,mrr,ndcg_at_k,ragas_id_precision,ragas_id_recall,required_fact_recall,forbidden_fact_hits,answer_context_overlap,ragas_exact_match,ragas_bleu,ragas_rouge,llm_judge_groundedness,llm_judge_reference_alignment,llm_judge_completeness,llm_judge_average
0,bm25_only,BM25Retriever,groq:llama-3.3-70b-versatile,0.833,0.542,1.0,1.0,0.705,0.5,0.542,0.771,0.0,0.845,0.0,0.36,0.592,4.75,4.5,4.25,4.5


Weakest fetch query: How can analysts identify chronic late payers?
Weakest rank query: Which field represents invoices more than 90 days overdue?
Weakest answer query: For chronic late payer reporting, which job should I inspect first?
This notebook now evaluates a BM25-only RAG pipeline with explicit ground truth at fetch, rank, and answer level using manual, Ragas, and LLM-judge metrics.
